In [7]:
from helper import DataLoader
from models.SimpleGCN import SimpleGCN 
import torch
from sklearn.metrics import root_mean_squared_error

In [8]:
DATAPATH = "../data"
RATING_DATA = DATAPATH + "/train_ratings.csv"
TBR_DATA = DATAPATH + "/train_tbr.csv"

In [9]:
data_manager = DataLoader.DataLoader(data_dir=DATAPATH)
train_df, valid_df = data_manager.load_and_split()

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
edge_index, edge_weights = data_manager.get_graph_tensors(device)

In [ ]:
EPOCHS = 200

# Initialize Model using dimensions from the manager
model = SimpleGCN.SimpleGCN(
    num_users=data_manager.num_users,
    num_items=data_manager.num_items,
    embedding_dim=64,
    num_layers=1
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-2,
)
criterion = torch.nn.MSELoss()

# --- 2. PREDICTION WRAPPER ---
def predict_ratings(sids, pids):
    model.eval()
    with torch.no_grad():
        s_tensor = torch.tensor(sids, dtype=torch.long).to(device)
        p_tensor = torch.tensor(pids, dtype=torch.long).to(device)
        # Return predictions scaled back to 1-5 range
        return model(s_tensor, p_tensor, edge_index, edge_weights).cpu().numpy() * 5.0

# --- 3. TRAINING LOOP ---
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    
    # Train on the existing edges
    preds = model(edge_index[0], edge_index[1], edge_index, edge_weights)
    loss = criterion(preds, edge_weights)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        # Evaluate using RMSE on validation set
        val_preds = predict_ratings(valid_df["sid"].values, valid_df["pid"].values)
        val_rmse = root_mean_squared_error(valid_df["rating"].values, val_preds)
        print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Val RMSE: {val_rmse:.4f}")


Epoch 0 | Loss: 0.6260 | Val RMSE: 3.9432
Epoch 10 | Loss: 0.2054 | Val RMSE: 1.9251
Epoch 20 | Loss: 0.1307 | Val RMSE: 1.6177
Epoch 30 | Loss: 0.0901 | Val RMSE: 1.4829
Epoch 40 | Loss: 0.0615 | Val RMSE: 1.2718
Epoch 50 | Loss: 0.0512 | Val RMSE: 1.1458
Epoch 60 | Loss: 0.0450 | Val RMSE: 1.0786
Epoch 70 | Loss: 0.0418 | Val RMSE: 1.0297
Epoch 80 | Loss: 0.0390 | Val RMSE: 0.9954
Epoch 90 | Loss: 0.0373 | Val RMSE: 0.9712
Epoch 100 | Loss: 0.0362 | Val RMSE: 0.9558
Epoch 110 | Loss: 0.0354 | Val RMSE: 0.9458
Epoch 120 | Loss: 0.0349 | Val RMSE: 0.9401
Epoch 130 | Loss: 0.0347 | Val RMSE: 0.9368
Epoch 140 | Loss: 0.0346 | Val RMSE: 0.9350
